In [2]:
"""
ADNI Baseline Data Preprocessing Script
- Filters baseline visits
- Selects and engineers features
- Handles missing values
- Saves clean dataset to CSV
"""

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load data
df = pd.read_csv('ADNIMERGE_25Apr2025.csv')

# Filter to baseline visits only
df_bl = df[df['VISCODE'] == 'bl'].copy()

# Select relevant features
features = ['PTID', 'AGE', 'APOE4', 'Hippocampus', 'ICV', 'MMSE', 'CDRSB', 'DX']
df_bl = df_bl[features]

# Show distribution of diagnosis labels
print("Diagnosis distribution:\n", df['DX'].value_counts(dropna=False))

# Normalize hippocampal volume
df_bl['Hippocampus_ICV'] = df_bl['Hippocampus'] / df_bl['ICV']

# Check missing values in key clinical scores
missing_percent = df_bl[['MMSE', 'CDRSB']].isna().mean() * 100
print("Missing percentage for MMSE and CDRSB:\n", missing_percent)

# Impute MMSE and CDRSB missing values using median within each DX group
for var in ['MMSE', 'CDRSB']:
    df_bl[var] = df_bl.groupby('DX')[var].transform(lambda x: x.fillna(x.median()))

# Verify remaining missing by diagnosis
missing_by_dx = df_bl.groupby('DX')[['MMSE', 'CDRSB']].apply(lambda x: x.isna().sum())
print("Remaining missing values per diagnosis group:\n", missing_by_dx)

# Display APOE4 distribution before imputation
print("Unique APOE4 values (before encoding):\n", df_bl['APOE4'].value_counts(dropna=False))

# Impute APOE4 using mode within each DX group
df_bl['APOE4'] = df_bl.groupby('DX')['APOE4'].transform(lambda x: x.fillna(x.mode()[0]))

# Drop rows missing critical normalized volume feature
df_bl_clean = df_bl.dropna(subset=['Hippocampus_ICV'])

# Show stats on data retained
total_retained = len(df_bl_clean)
print(f"Retained {total_retained}/{len(df_bl)} samples ({total_retained / len(df_bl):.1%})")
print("Diagnosis distribution in cleaned data:\n", df_bl_clean['DX'].value_counts())

# Check for any remaining missing values
total_na = df_bl_clean.isna().sum().sum()
print(f"Total number of NA values in cleaned data: {total_na}")

# Investigate rows with any missing values (should be 0 if properly handled)
missing_rows = df_bl_clean[df_bl_clean.isnull().any(axis=1)]
if not missing_rows.empty:
    print("Sample rows with missing data:\n", missing_rows.head())
    print("Missing columns per row:\n", missing_rows.isnull().sum(axis=1).value_counts())

# Final filtering: remove any remaining NAs in critical variables
critical_vars = ['DX', 'APOE4', 'Hippocampus_ICV']
df_bl_final = df_bl_clean.dropna(subset=critical_vars)

# Fill remaining AGE NAs with median
df_bl_final['AGE'] = df_bl_final['AGE'].fillna(df_bl_final['AGE'].median())

# Final summary
print(f"Final sample size: {len(df_bl_final)}")
print("Remaining missing:\n", df_bl_final.isnull().sum())

# Save final cleaned dataset
output_path = '/content/ADNI_baseline_clean.csv'
df_bl_final.to_csv(output_path, index=False)
print(f"Saved {len(df_bl_final)} rows to: {output_path}")


Diagnosis distribution:
 DX
MCI         4989
NaN         4963
CN          4020
Dementia    2449
Name: count, dtype: int64
Missing percentage for MMSE and CDRSB:
 MMSE     0.041152
CDRSB    0.000000
dtype: float64
Remaining missing values per diagnosis group:
           MMSE  CDRSB
DX                   
CN           0      0
Dementia     0      0
MCI          0      0
Unique APOE4 values (before encoding):
 APOE4
0.0    1199
1.0     803
NaN     217
2.0     211
Name: count, dtype: int64
Retained 2081/2430 samples (85.6%)
Diagnosis distribution in cleaned data:
 DX
MCI         928
CN          800
Dementia    338
Name: count, dtype: int64
Total number of NA values in cleaned data: 63
Sample rows with missing data:
             PTID   AGE  APOE4  Hippocampus        ICV  MMSE  CDRSB   DX  \
4191  036_S_4740  88.3    NaN       6480.0  1678780.0   NaN    NaN  NaN   
4711  016_S_4575  62.1    NaN       7228.0  1268820.0   NaN    NaN  NaN   
5436  094_S_4459  68.1    NaN       7798.0  1375170.0 

<ipython-input-2-1d358aa59afa>:14: DtypeWarning: Columns (19,20,21,50,51,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('ADNIMERGE_25Apr2025.csv')
<ipython-input-2-1d358aa59afa>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bl_final['AGE'] = df_bl_final['AGE'].fillna(df_bl_final['AGE'].median())
